In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# If True, test is running on Colab. Otherwise, test if assumed to be offline.
TEST_ON_COLAB = False
FOLDERNAME = None  # only used if TEST_ON_COLAB is True

assert not (FOLDERNAME is None and TEST_ON_COLAB), "FOLDERNAME has to be set if TEST_ON_COLAB is True"

In [ ]:
import torch

# Set the device to be used for training (cuda:0 if there is a GPU available, otherwise cpu)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Atividade de programação 4 (AP4)
## Vision Transformer (ViT)
### Valor: 25 pontos

Responda as questões indicadas e preencha os campos indicados com o tag `TODO`.

Recomenda-se executar este notebook em um ambiente com GPU (Ambiente de execução → Alterar o tipo de ambiente de execução).

Para essa atividade, você deverá utilizar o mesmo dataset de classificação utilizado na AP3. Caso não tenha realizado essa atividade, pesquise um dataset de imagens de sua preferência (exceto mnist) utilizando o site `https://openml.org/`. O dataset será baixado através de função `fetch_openml` da biblioteca `scikit-learn`. O dataset deve apresentar a mesma estrutura de pixels/labels ($X$, $y$) vistos em exemplos de aula. Caso não esteja assim, faça as devidas transformações.

In [ ]:
# TODO: Set the dataset ID
DATASET_OPENML_ID = None
##########################

assert DATASET_OPENML_ID is not None, "DATASET_OPENML_ID is not set"

Utilizaremos uma semente para garantir a reprodutibilidade dos resultados.

In [ ]:
SEED = 42

### Base de dados

O código a seguir realiza, se necessário, o download da base de dados. Em seguida, as amostras são carregadas em formato matricial.

In [ ]:
# Create dataset directory
import os

if TEST_ON_COLAB:
    # This mounts your Google Drive to the Colab VM.
    from google.colab import drive

    drive.mount("/content/drive")
    cache_dir = f"/content/drive/My Drive/{FOLDERNAME}/dataset/{DATASET_OPENML_ID}"
else:
    cache_dir = f"dataset/{DATASET_OPENML_ID}"

os.makedirs(cache_dir, exist_ok=True)

# Fetching the dataset
from sklearn.datasets import fetch_openml
import numpy as np

X_file_path = f"{cache_dir}/X.npy"
y_file_path = f"{cache_dir}/y.npy"

# Check if the dataset files already exist
if not (os.path.exists(X_file_path) and os.path.exists(y_file_path)):
    # Fetch the dataset where X is the data and y is the target
    X, y = fetch_openml(DATASET_OPENML_ID, as_frame=False, cache=True, return_X_y=True)

    # Save the dataset as numpy arrays
    np.save(X_file_path, X.astype(np.float32))
    np.save(y_file_path, y)
    print(f"{DATASET_OPENML_ID} dataset downloaded and saved successfully to {cache_dir}.")
else:
    X = np.load(X_file_path, allow_pickle=True)
    y = np.load(y_file_path, allow_pickle=True)
    print(f"{DATASET_OPENML_ID} dataset already exists in {cache_dir}. Skipping download.")

# Cast to 32-bits float number
X = X.astype(np.float32)

print(X.shape)
print(X.min(), X.max(), X.dtype)

A seguir, são exibidas algumas amostras do dataset. Ajuste a variável `image_shape` de acordo com as dimensões das amostras da sua base de dados.

In [ ]:
# TODO: Set the image shape. ##################
image_shape = None # e.g. (28, 28), (32, 32, 3)
###############################################

assert image_shape is not None, "image_shape is not set"
assert len(image_shape) == 2 or len(image_shape) == 3, "image_shape should be a tuple of 2 or 3 elements"

import matplotlib.pyplot as plt

# Visualize some examples from the dataset.
# We show a few examples of training images from each class.
classes = [int(class_id) for class_id in np.unique(y)]
num_classes = len(classes)
samples_per_class = 7
for cls in classes:
    idxs = np.flatnonzero(y == str(cls))
    idxs = np.random.choice(idxs, samples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt_idx = i * num_classes + cls + 1
        plt.subplot(samples_per_class, num_classes, plt_idx)
        plt.imshow(X[idx].reshape(image_shape).astype('uint8'), cmap='gray')
        plt.axis('off')
        if i == 0:
            plt.title(cls)
plt.show()

**1. (1 ponto)** Agora, você precisa particionar o conjunto de dados em dois subconjuntos: (i) treino + validação (train+val) e (ii) teste (test).

Utilize a função `train_test_split` do módulo `sklearn.model_selection` para dividir o conjunto de dados (`X` e `y`): 90% dos dados para o subconjunto train+val e 10% para o subconjunto test.

In [ ]:
from sklearn.model_selection import train_test_split

# Convert labels from string to integer
y = y.astype(int)

# Set the random seed for reproducibility
np.random.seed(SEED)

# TODO: Split the dataset into train and test sets
X_train_val, X_test, y_train_val, y_test = None
##################################################

assert X_train_val is not None, "X_train_val is not set"
assert y_train_val is not None, "y_train_val is not set"
assert X_test is not None, "X_test is not set"
assert y_test is not None, "y_test is not set"

print(f"Train+val set size: {len(X_train_val)}")
print(f"Test set size: {len(X_test)}")

**2. (1 ponto)** Agora, você precisa separar o conjunto de treino (train) do conjunto de validação (val).

Utilize novamente a função `train_test_split` do módulo `sklearn.model_selection` para dividir o conjunto de dados (`X_train_val` e `y_train_val`): 90% dos dados para o subconjunto train e 10% para o subconjunto val.

In [ ]:
# TODO: Split further the train+val dataset into train and validation sets
X_train, X_val, y_train, y_val = None
##########################################################################

assert X_val is not None, "X_val is not set"
assert y_val is not None, "y_val is not set"

print(f"Train set size: {len(X_train)}")
print(f"Val set size: {len(X_val)}")

Transformers têm custo computacional elevado, e nesta atividade vários modelos serão treinados. Por isso, limitamos o número de amostras de cada partição, mantendo a proporção das classes. Se o seu dataset tiver menos amostras do que esses limites, ele será usado por inteiro.

In [ ]:
MAX_TRAIN_SAMPLES = 5000
MAX_EVAL_SAMPLES = 1000


def limit_samples(X, y, max_samples):
    if len(X) <= max_samples:
        return X, y
    X, _, y, _ = train_test_split(X, y, train_size=max_samples, random_state=SEED, stratify=y)
    return X, y


X_train, y_train = limit_samples(X_train, y_train, MAX_TRAIN_SAMPLES)
X_val, y_val = limit_samples(X_val, y_val, MAX_EVAL_SAMPLES)
X_test, y_test = limit_samples(X_test, y_test, MAX_EVAL_SAMPLES)

print(f"Train set size: {len(X_train)}")
print(f"Val set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

### Implementação em Pytorch

#### Modelo Vision Transformer

Um Vision Transformer divide a imagem em *patches* (pedaços) de tamanho fixo, projeta cada patch em um vetor (*token*) e processa a sequência de tokens com um Transformer encoder. Um token extra, chamado `[CLS]`, reúne a informação da imagem e alimenta a camada de classificação.

```
@inproceedings{dosovitskiy2021vit,
author = {Dosovitskiy, Alexey and Beyer, Lucas and Kolesnikov, Alexander and others},
title = {An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale},
booktitle = {International Conference on Learning Representations (ICLR)},
year = {2021}
}
```

Adotamos o modelo `facebook/deit-tiny-patch16-224`, da biblioteca Hugging Face Transformers: um ViT pequeno (cerca de 5 milhões de parâmetros), com patches de 16x16 pixels e entrada de 224x224 pixels, pré-treinado na ImageNet.

A camada de classificação é ajustada de acordo com o número de saídas esperadas (`n_output`). Assim como na AP3, é possível treinar o modelo do zero (`pretrained=False`) ou fazer o ajuste fino de um modelo pré-treinado, em todas as camadas ou apenas na última (`freeze=True`). O parâmetro `dropout_rate` controla a regularização por dropout dentro do Transformer.

In [ ]:
from transformers import ViTConfig, ViTForImageClassification

MODEL_NAME = "facebook/deit-tiny-patch16-224"


class ViT(torch.nn.Module):
    def __init__(self, n_output, pretrained=True, dropout_rate=0.0, freeze=False):
        super(ViT, self).__init__()

        # The eager implementation is required to read the attention weights
        kwargs = {"hidden_dropout_prob": dropout_rate, "attn_implementation": "eager"}
        if pretrained:
            self.backbone = ViTForImageClassification.from_pretrained(
                MODEL_NAME, num_labels=n_output, ignore_mismatched_sizes=True, **kwargs
            )
        else:
            config = ViTConfig.from_pretrained(MODEL_NAME, num_labels=n_output, **kwargs)
            self.backbone = ViTForImageClassification(config)

        # Freeze all parameters, except the ones of the classification layer
        if freeze:
            for param in self.backbone.vit.parameters():
                param.requires_grad = False

    def forward(self, x):
        return self.backbone(pixel_values=x).logits

    def attentions(self, x):
        """Returns a tuple with the attention weights of each layer, of shape (N, heads, tokens, tokens)."""
        return self.backbone(pixel_values=x, output_attentions=True).attentions


# Define the model
model = ViT(n_output=num_classes)
model.to(device)

A célula a seguir mostra a implementação da classe Dataset para gerenciamento dos dados. As imagens são convertidas para 3 canais, redimensionadas para 224x224 pixels e normalizadas com as estatísticas da ImageNet, conforme esperado pelos modelos pré-treinados.

In [ ]:
from torchvision.datasets import VisionDataset
from torchvision.transforms import v2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

preprocess = v2.Compose([
    v2.Resize((224, 224), antialias=True),
    v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class Dataset(VisionDataset):
    def __init__(self, X, y, image_shape, transform=None):
        super(Dataset, self).__init__("", transform=transform)

        # Determine height, width, and number of channels
        H, W = image_shape[:2] # height, width
        C = 1 if len(image_shape) == 2 else image_shape[2] # channels

        # Reshape the data to (N, H, W, C)
        self.X = X.reshape(-1, H, W, C)

        # Transpose the data to (N, C, H, W) format
        self.X = np.transpose(self.X, (0, 3, 1, 2))

        # Replicate first channel to have 3 channels (enable pretrained models)
        if C == 1:
            self.X = np.repeat(self.X, 3, axis=1)

        # Pixel values are mapped to the range [0, 1]
        self.scale = 255.0 if self.X.max() > 1 else 1.0

        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        image = torch.from_numpy(self.X[index] / self.scale).float()
        label = int(self.y[index])
        if self.transform is not None:
            image = self.transform(image)
        return image, label

# Create the datasets
datasets = {
    "train": Dataset(X_train, y_train, image_shape, preprocess),
    "val": Dataset(X_val, y_val, image_shape, preprocess),
    "test": Dataset(X_test, y_test, image_shape, preprocess)
}

# Statiscs of the dataset
print(f"Training dataset shape: {datasets['train'].X.shape}")
print(f"Validation dataset shape: {datasets['val'].X.shape}")
print(f"Test dataset shape: {datasets['test'].X.shape}")

O carregamento dos dados em batch é implementado de acordo com objetos da classe `DataLoader`:

In [ ]:
from torch.utils.data import DataLoader

batch_size = 64

assert batch_size is not None, "batch_size is not set"

dataloaders = {
    "train": DataLoader(datasets["train"], batch_size=batch_size, shuffle=True),
    "val": DataLoader(datasets["val"], batch_size=batch_size, shuffle=False),
    "test": DataLoader(datasets["test"], batch_size=batch_size, shuffle=False)
}

**3. (4 pontos)** O primeiro passo de um ViT é transformar a imagem em uma sequência de patches. Implemente a função `patchify`, que recebe um batch de imagens de formato `(N, C, H, W)` e retorna um tensor de formato `(N, T, C * P * P)`, em que `P` é o tamanho do patch e `T` é o número de patches. Os patches devem seguir a ordem de leitura da imagem (da esquerda para a direita, de cima para baixo), e cada patch deve ser achatado na ordem canal, linha, coluna.

O teste a seguir compara o seu resultado com a camada de *patch embedding* do modelo, que faz o mesmo cálculo com uma convolução.

Dica: utilize `reshape` e `permute`.

In [ ]:
def patchify(images, patch_size):
    """
    Split a batch of images into a sequence of flattened patches.

    Args:
        images: Tensor of shape (N, C, H, W).
        patch_size: The height and width of each patch.

    Returns:
        Tensor of shape (N, T, C * patch_size * patch_size).
    """
    # TODO: Implement the function
    patches = None
    ###############################
    return patches


images, _ = next(iter(dataloaders["test"]))
images = images.to(device)
patches = patchify(images, 16)

assert patches is not None, "patches is not set"
print(f"Input shape: {tuple(images.shape)}")
print(f"Patches shape: {tuple(patches.shape)}")

# A linear projection of the patches must match the convolution of the model
projection = model.backbone.vit.embeddings.patch_embeddings.projection
with torch.no_grad():
    expected = projection(images).flatten(2).transpose(1, 2)
    tokens = patches @ projection.weight.flatten(1).T + projection.bias
print(f"Maximum difference: {(tokens - expected).abs().max().item():.2e}")
assert torch.allclose(tokens, expected, atol=1e-3), "patchify does not match the patch embedding of the model"

### Treinamento do modelo

Antes de realizar o treinamento, configuramos a biblioteca `pytorch` para garantir replicabilidade dos experimentos:

In [ ]:
import torch

# Set the random seed for reproducibility
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

A seguir, tem-se a implementação do treinamento do modelo.

In [ ]:
import time

def train(model, dataloaders, learning_rate=0.0001, weight_decay=0.0, num_epochs=3, checkpoint="model.pt"):
    """
    Train the model.

    Args:
        model: The model to train.
        dataloaders: The dataloaders.
        learning_rate: The learning rate.
        weight_decay: The weight decay.
        num_epochs: The number of epochs.
        checkpoint: The path to the checkpoint.
    """

    # Cross-entropy loss
    criterion = torch.nn.CrossEntropyLoss()

    # Optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Initialize the best (validation) accuracy
    best_acc = -1
    best_epoch = -1

    # Training loss and validation accuracy history
    train_loss_history = []
    val_acc_history = []

    # Initialize the time
    since = time.time()

    # Training loop
    for epoch in range(1, num_epochs + 1):
        print(f"Epoch {epoch}/{num_epochs}")
        print("-" * 10)

        # Each epoch has a training, validation, and test phase
        for phase in ["train", "val"]:
            if phase == "train":
                model.train()  # Set model to training mode
            else:
                model.eval()  # Set model to evaluate mode

            # Reset loss and counter of correct predictions
            running_loss = 0.0
            running_corrects = 0

            # Iterate over data
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # zero the parameter gradients
                optimizer.zero_grad()

                # forward
                # track history if only in train
                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # backward + optimize only if in training phase
                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                # statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data).item()

            num_samples = len(dataloaders[phase].dataset)
            epoch_loss = running_loss / num_samples
            epoch_acc = running_corrects / num_samples

            # Update history
            if phase == "train":
                train_loss_history.append(epoch_loss)
            if phase == "val":
                val_acc_history.append(epoch_acc)

            # Calculate loss and accuracy
            print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            # Save the (current) best model (based on the validation accuracy)
            if phase == "val" and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_epoch = epoch
                torch.save(model.state_dict(), checkpoint)

    time_elapsed = time.time() - since
    print(f"Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    print(f"Best validation accuracy: {best_acc:.4f} at epoch {best_epoch}")

    return train_loss_history, val_acc_history, best_epoch, time_elapsed

# Train the model for 3 epochs
train_loss_history, val_acc_history, best_epoch, _ = train(model, dataloaders)

O treinamento produziu as seguintes curvas de aprendizado:

In [ ]:
# Plot training curve
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots()

# Training loss curve
epochs = range(1, len(train_loss_history) + 1)
ax1.plot(epochs, train_loss_history, label='Training Loss', color='tab:red')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss', color='tab:red')
ax1.tick_params(axis='y', labelcolor='tab:red')
ax1.set_xticks(epochs)

# Validation accuracy curve
ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis
ax2.plot(epochs, val_acc_history, label='Validation Accuracy', color='tab:blue')
ax2.set_ylabel('Accuracy', color='tab:blue')
ax2.tick_params(axis='y', labelcolor='tab:blue')

# Highlight the best epoch
ax2.axvline(best_epoch, linestyle='dashed', label='Best Epoch', color='gray')

# Additional information
# Legend
fig.tight_layout()
fig.legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=3)

plt.title("Training Curve")

plt.show()

### Teste

A partição de teste é utilizada para avaliar o modelo treinado:

In [ ]:
@torch.no_grad()
def test(model, checkpoint="model.pt"):
    """
    Test the model.

    Args:
        model: The model to test.
        checkpoint: The path to the model checkpoint
    """
    params = torch.load(checkpoint, map_location=device)
    model.load_state_dict(params)
    model.eval()

    pred_labels = []
    true_labels = []

    # Iterate over data
    for inputs, labels in dataloaders["test"]:
        inputs = inputs.to(device)
        outputs = model(inputs)
        pred_labels_batch = torch.argmax(outputs, dim=1)
        pred_labels.append(pred_labels_batch)
        true_labels.append(labels)

    pred_labels = torch.cat(pred_labels, dim=0).cpu().numpy()
    true_labels = torch.cat(true_labels, dim=0).numpy()

    return pred_labels, true_labels

# Test the model
pred_labels, true_labels = test(model)

Os resultados obtidos são exibidos em formato de matriz de confusão e métricas de avaliação (relatório de classificação):

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

def print_classification_report(true_labels, pred_labels, num_classes):
    """
    Print the classification report.
    """
    # Compute classification report
    class_report = classification_report(true_labels, pred_labels, target_names=[str(class_id) for class_id in range(num_classes)])
    print("\nClassification Report:")
    print(class_report)


def plot_confusion_matrix(true_labels, pred_labels, num_classes):
    """
    Plot the confusion matrix.
    """
    # Compute confusion matrix
    cm = confusion_matrix(true_labels, pred_labels, labels=[class_id for class_id in range(num_classes)])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[class_id for class_id in range(num_classes)])
    disp.plot()
    # plt.show()


print_classification_report(true_labels, pred_labels, num_classes)
plot_confusion_matrix(true_labels, pred_labels, num_classes)

**4. (6 pontos)** Investigue a importância do pré-treinamento para um ViT. Treine e teste o modelo nas três configurações abaixo, usando a mesma quantidade de épocas. Guarde a acurácia de teste em `test_accuracies` e o tempo de treinamento em `train_times`. Cada configuração deve salvar o seu próprio *checkpoint*, com o nome indicado em `checkpoint`.

Em seguida, responda na célula de texto: qual configuração obteve o melhor resultado? Por que um ViT treinado do zero com poucos dados tende a ter desempenho inferior ao de uma CNN nas mesmas condições?

In [ ]:
from sklearn.metrics import accuracy_score

setups = {
    "Ajuste fino completo": {"pretrained": True, "freeze": False, "learning_rate": 0.0001, "checkpoint": "vit_full.pt"},
    "Apenas a última camada": {"pretrained": True, "freeze": True, "learning_rate": 0.001, "checkpoint": "vit_head.pt"},
    "Treino do zero": {"pretrained": False, "freeze": False, "learning_rate": 0.0001, "checkpoint": "vit_scratch.pt"},
}
test_accuracies = {}
train_times = {}

for name, setup in setups.items():
    # TODO: Train and test the model
    pass
    ################################

assert len(test_accuracies) == len(setups), "test_accuracies is not complete"

plt.bar(test_accuracies.keys(), test_accuracies.values())
plt.ylabel('Test Accuracy')
plt.title('ViT: effect of pre-training')
plt.show()

**Resposta:**

**5. (5 pontos)** Compare o ViT com a CNN da AP3. Treine a `Resnet18` abaixo, pré-treinada e sem congelamento, com as mesmas partições e a mesma quantidade de épocas do ajuste fino completo da questão anterior. Complete a tabela com a acurácia de teste e o tempo de treinamento.

Em seguida, responda na célula de texto: qual modelo teve melhor acurácia? Qual tem mais parâmetros? Qual foi mais rápido? Os resultados confirmam o que você esperava, considerando a quantidade de dados utilizada?

In [ ]:
from torchvision import models


class Resnet18(torch.nn.Module):
    def __init__(self, n_output):
        super(Resnet18, self).__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.backbone.fc = torch.nn.Linear(self.backbone.fc.in_features, n_output)

    def forward(self, x):
        return self.backbone(x)


def count_parameters(model):
    return sum(param.numel() for param in model.parameters())


resnet = Resnet18(n_output=num_classes).to(device)

# TODO: Train and test the ResNet18
resnet_accuracy = None
resnet_time = None
###################################

assert resnet_accuracy is not None, "resnet_accuracy is not set"
assert resnet_time is not None, "resnet_time is not set"

vit = ViT(n_output=num_classes)
print(f"{'Modelo':<12}{'Parâmetros':>14}{'Acurácia':>12}{'Tempo (s)':>12}")
print(f"{'ViT':<12}{count_parameters(vit):>14,}{test_accuracies['Ajuste fino completo']:>12.4f}{train_times['Ajuste fino completo']:>12.1f}")
print(f"{'ResNet18':<12}{count_parameters(resnet):>14,}{resnet_accuracy:>12.4f}{resnet_time:>12.1f}")

**Resposta:**

**6. (5 pontos)** Uma vantagem dos Transformers é que os pesos de atenção indicam quais tokens influenciaram cada representação. Implemente a função `attention_map`, que retorna a atenção do token `[CLS]` para os patches da imagem na **última camada**, com a média entre as cabeças de atenção. O método `model.attentions(x)` retorna uma tupla com um tensor de formato `(N, heads, tokens, tokens)` por camada, e o token `[CLS]` é o primeiro da sequência.

A célula seguinte exibe o resultado para exemplos classificados corretamente e incorretamente pelo modelo de ajuste fino completo. Responda na célula de texto: as regiões destacadas fazem sentido para a classe da imagem? Há diferença entre os acertos e os erros?

In [ ]:
@torch.no_grad()
def attention_map(model, image):
    """
    Compute the attention of the [CLS] token over the image patches.

    Args:
        model: The ViT model.
        image: Tensor of shape (3, 224, 224).

    Returns:
        Tensor of shape (14, 14), with one value per patch.
    """
    model.eval()
    # TODO: Implement the function
    result = None
    ###############################
    return result


best_vit = ViT(n_output=num_classes).to(device)
pred_labels, true_labels = test(best_vit, "vit_full.pt")

assert attention_map(best_vit, datasets["test"][0][0]) is not None, "attention_map is not implemented"


def show_attention(indices, title):
    mean = torch.tensor(IMAGENET_MEAN).reshape(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).reshape(3, 1, 1)
    fig, axes = plt.subplots(2, len(indices), figsize=(3 * len(indices), 6), squeeze=False)
    for column, index in enumerate(indices):
        image = datasets["test"][index][0]
        attention = attention_map(best_vit, image).cpu()
        axes[0, column].imshow((image * std + mean).clamp(0, 1).permute(1, 2, 0))
        axes[0, column].set_title(f"Real: {true_labels[index]}, Predito: {pred_labels[index]}")
        axes[1, column].imshow(attention, cmap="viridis")
        axes[0, column].axis("off")
        axes[1, column].axis("off")
    fig.suptitle(title)
    plt.show()


correct = np.flatnonzero(pred_labels == true_labels)[:4]
wrong = np.flatnonzero(pred_labels != true_labels)[:4]
show_attention(correct, "Acertos")
if len(wrong) > 0:
    show_attention(wrong, "Erros")

**Resposta:**

**7. (3 pontos)** Neste exercício final, você trabalhará para obter o melhor ViT variando dois hiperparâmetros: taxa de aprendizado e taxa de dropout. Você deverá treinar os diferentes modelos e utilizar, no teste, aquele que resultar na maior acurácia no conjunto de **VALIDAÇÃO**.

Dica: salve um *checkpoint* para cada configuração. A função `train` retorna o histórico de acurácia na validação.

In [ ]:
best_checkpoint = None
best_val_acc = -1
best_dropout_rate = None

learning_rate_vals = [0.0001, 0.00003]
dropout_rate_vals = [0.0, 0.1]

for learning_rate in learning_rate_vals:
    for dropout_rate in dropout_rate_vals:
        # TODO: Train the models in different setups
        pass
        ##############################################


# Test the best model and report results
assert best_checkpoint is not None

print(f"Best checkpoint={best_checkpoint} (val. acc.={best_val_acc:.3f})")

model = ViT(n_output=num_classes, dropout_rate=best_dropout_rate).to(device)
pred_labels, true_labels = test(model, checkpoint=best_checkpoint)
print_classification_report(true_labels, pred_labels, num_classes)
plot_confusion_matrix(true_labels, pred_labels, num_classes)